# 3. Floor-adjusted + EIP-8372

Reproduces demand-calibrated CPSB, the state-limit scale, integer accounting, central simulation, elasticity sensitivity, state-utilization diagnostics, state-demand caps, and resource pulses in [the report](../../markdowns/eip8279_floor_calibration_report.md).

Use the report's EIP-8372 model specification checked on 9 September 2026. The calibrated constants are empirical model choices. Each propagation allocation inherits its floor and common limit from Notebook 2; its 35-day calibration remains fixed under all alternative elasticity vectors.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/shared_fee/replay.py").is_file())
for directory in (ROOT / "src", ROOT / "scripts", ROOT / "scripts/shared_fee"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from publication_workflow import (
    DATA, REPORT, LABELS, baseline_anchor, read_table, require_files,
    run_stage, shared_display, source_snapshot, verify_sources,
)
DATA.mkdir(parents=True, exist_ok=True)
(ROOT / "plots").mkdir(exist_ok=True)
REUSE = os.environ.get("ONE_DIMENSIONAL_REUSE_OUTPUTS", "0") == "1"
REFRESH_XATU = os.environ.get("ONE_DIMENSIONAL_REFRESH_XATU", "0") == "1"
before = source_snapshot()
pd.set_option("display.max_columns", 24)
print("Repository:", ROOT)
print("Reuse generated outputs:", REUSE, "| Refresh Xatu inputs:", REFRESH_XATU)

## Derive the state-price scale

Let $c_0$ denote state-budget-matched CPSB and $c=kc_0$ the demand-calibrated value. Solve the regular branch (execution plus data/floor gas) at its target to obtain $b_{\mathrm{regular}}$. Let $b_{\mathrm{state}}$ clear state demand at $k=1$. Then $k^*=b_{\mathrm{state}}/b_{\mathrm{regular}}$ aligns the two unshocked normalized branches.

After CPSB rounding, the integer percentage scale and counters are:

$$
s=\lfloor100c/c_0\rfloor,\quad
L_{\mathrm{state,raw}}=\lfloor L_{\mathrm{shared}}s/100\rfloor,\quad
g_{\mathrm{state,normalized}}=\lfloor100g_{\mathrm{state,raw}}/s\rfloor.
$$

State demand pays the raw state price $cb$. The controller compares regular gas with normalized state gas. Scaling CPSB and the raw state limit together preserves the physical byte capacity. The calibration assumes that regular demand does not depend directly on the state-price scale.

In [ ]:
import run_eip8372_calibration as normalized_runner
specs, base, adjusted_source, demand, metering = normalized_runner.inputs()
derived = normalized_runner.calibration_table(specs, base, adjusted_source)
display(derived[["propagation_time_s", "shared_target", "shared_limit", "floor_rate",
                 "baseline_cpsb", "cpsb", "effective_scale", "state_gas_limit_scale",
                 "raw_state_limit", "equilibrium_fee_wei",
                 "equilibrium_regular_utilization", "equilibrium_state_utilization"]])
assert len(derived) == 5
assert derived.byte_capacity_relative_error.abs().max() < 1e-5
assert (derived.equilibrium_regular_utilization - 1).abs().max() < 1e-5
assert (derived.equilibrium_state_utilization - 1).abs().max() < 1e-5

## Simulate the fixed calibration

The runner executes 20 normalized-state cases (five calibrations × four complete elasticity vectors), a deterministic rounding control, and 40 matched frozen EIP-7999 references. The reference replays verify parity with the existing EIP-7999 results. All stochastic cases use the same 32 paths and 7,200/50,400 burn-in/measurement split.

In [ ]:
run_stage("run_eip8372_calibration.py", outputs=[
    "eip8372/calibration.csv", "eip8372/normalized_outcomes.csv", "eip8372/normalized_paths.csv",
    "eip8372/fixed_7999_outcomes.csv", "eip8372/fixed_7999_paths.csv",
    "eip8372/paired_gains.csv", "eip8372/paired_gain_paths.csv",
    "eip8372/unshocked_control.csv", "eip8372/manifest.json",
], reuse=REUSE)
cal = read_table("eip8372/calibration.csv")
normalized = read_table("eip8372/normalized_outcomes.csv")
manifest = json.loads((DATA / "eip8372/manifest.json").read_text())
assert manifest["shape"] == [32, 57_600, 4]
assert manifest["calibration_window_days"] == 35
assert len(normalized) == 20
for column in ["cpsb", "state_gas_limit_scale", "equilibrium_fee_wei"]:
    np.testing.assert_allclose(
        cal.sort_values("propagation_time_s")[column],
        derived.sort_values("propagation_time_s")[column], rtol=1e-12)
central = normalized[normalized.window_days.eq(35)].sort_values("propagation_time_s")
display(pd.DataFrame({
    "Propagation (s)": central.propagation_time_s,
    "Execution (M)": central.metered_execution_gas / 1e6,
    "Regular target utilization (%)": 100 * central.regular_utilization,
    "Normalized state utilization (%)": 100 * central.state_utilization,
    "State growth (GiB/year)": central.annualized_state_growth_gib,
    "State as bottleneck (%)": 100 * central.state_binding_fraction,
    "At either limit (%)": 100 * central.hard_limit_fraction,
    "Fee variation": central.execution_price_variation,
}).reset_index(drop=True))
np.testing.assert_allclose(central.metered_execution_gas / 1e6,
                           [166.9, 177.9, 174.7, 165.3, 151.9], atol=0.05, rtol=0)
display(read_table("eip8372/unshocked_control.csv"))

## Frozen-calibration elasticity sensitivity and low-fee diagnostic

All three elasticities change together; CPSB, the state-limit scale, floor, limits, and recovered workload remain fixed at their central values for each propagation allocation. The four-setting diagnostic examines 60-/75-day vectors at 3.5/5.0 seconds, including fees relative to the state-clearing level and zero downward updates. It describes replay behavior without attributing a causal share to rounding.

In [ ]:
display(normalized[["window_days", "propagation_time_s", "equilibrium_fee_wei",
                    "metered_execution_gas", "annualized_state_growth_gib",
                    "regular_utilization", "state_utilization", "state_binding_fraction",
                    "hard_limit_fraction"]].sort_values(["window_days", "propagation_time_s"]))
run_stage("diagnose_eip8372_state_utilization.py", outputs=[
    "eip8372/state_utilization_diagnostics.csv", "eip8372/state_utilization_diagnostic_paths.csv",
    "eip8372/state_utilization_diagnostic_manifest.json",
], reuse=REUSE)
diagnostic = read_table("eip8372/state_utilization_diagnostics.csv")
display(diagnostic)
diagnostic_manifest = json.loads((DATA / "eip8372/state_utilization_diagnostic_manifest.json").read_text())
assert diagnostic_manifest["workload_sha256"] == manifest["workload_sha256"]
assert len(diagnostic) == 4

## State pricing and deployment-capacity appendix

The three-second example checks 64 KiB of new runtime code plus a 120-byte account contribution. It is a state-capacity calculation, not a complete transaction-feasibility test. Raw state-gas charges must be assessed against the correspondingly scaled raw limit.

In [ ]:
from shared_fee.optimization import BLOCKS_PER_YEAR, BYTES_PER_GIB
three = cal[cal.propagation_time_s.eq(3.)].iloc[0]
capacity = pd.DataFrame({
    "Configuration": ["200M / CPSB 1,530 reference", "Calibrated 3.0s"],
    "Maximum state bytes": [200e6 / 1530, three.raw_state_limit / three.cpsb],
})
capacity["State bytes at half limit"] = capacity["Maximum state bytes"] / 2
capacity["Annualized target (GiB/year)"] = (
    capacity["State bytes at half limit"] * BLOCKS_PER_YEAR / BYTES_PER_GIB)
display(capacity)
deployment_bytes = 64 * 1024 + 120
display(pd.DataFrame([{
    "deployment_bytes": deployment_bytes,
    "share_of_raw_state_limit_percent": 100 * deployment_bytes * three.cpsb / three.raw_state_limit,
    "fits_state_limit": deployment_bytes * three.cpsb <= three.raw_state_limit,
    "state_byte_price_gwei": three.cpsb * three.equilibrium_fee_wei / 1e9,
}]))

## Resource-pulse diagnostics

At four seconds, double one demand factor at onset and decay the additional component with a 120-block half-life. Compare against that design's identical no-pulse path over 600 event blocks. The runner records effective-price peaks and recovery against the same no-pulse trajectory; all other sampled shocks remain unchanged.

In [ ]:
run_stage("run_eip8372_stresses.py", outputs=[
    "eip8372/stress_outcomes.csv", "eip8372/stress_response_curve.csv", "eip8372/stress_manifest.json",
], reuse=REUSE)
stress = read_table("eip8372/stress_outcomes.csv")
assert len(stress) == 9
display(stress)
import make_eip8372_figures as figures
figures.stress_figure()
display(Image(filename=str(ROOT / "plots/shared_fee_eip8372_resource_pulses.png")))

## State-demand saturation at three seconds

Re-solve and replay the six cap settings with the three-second 35-day constants fixed. The retained empirical shock can still raise realized demand above the capped price-driven component. These sensitivities compare capped one-dimensional demand against fixed **unrestricted** EIP-7999 references. Notebook 4 reports the paired gains and their weekly dispersion.

In [ ]:
run_stage("run_eip8372_state_tail.py", outputs=[
    "eip8372/state_tail/normalized_outcomes.csv", "eip8372/state_tail/normalized_paths.csv",
    "eip8372/state_tail/three_second_comparison.csv",
    "eip8372/state_tail/paired_gains.csv", "eip8372/state_tail/paired_gain_paths.csv",
    "eip8372/state_tail/manifest.json",
], reuse=REUSE)
tail = read_table("eip8372/state_tail/normalized_outcomes.csv")
assert len(tail) == 6
assert len(read_table("eip8372/state_tail/normalized_paths.csv")) == 6 * 32
for column in ["cpsb", "state_gas_limit_scale", "shared_limit", "shared_target", "floor_rate"]:
    assert tail[column].nunique() == 1
display(tail[["state_demand_cap_label", "equilibrium_fee_wei", "metered_execution_gas",
              "annualized_state_growth_gib", "state_binding_fraction", "hard_limit_fraction"]])
figures.central_figure()
figures.elasticity_figure()
figures.three_mechanism_elasticity_figure()
display(Image(filename=str(ROOT / "plots/shared_fee_eip8372_central.png")))
display(Image(filename=str(ROOT / "plots/shared_fee_elasticity_execution_state.png")))
verify_sources(before)